In [ ]:
import os
import zipfile
import io
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# Configuration esthétique des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# =====================================================================
# 1. CHARGEMENT DIRECT DEPUIS VOS TÉLÉCHARGEMENTS (Répond au Commentaire 1 & 2)
# =====================================================================

print("--- CLIQUEZ CI-DESSOUS POUR CHARGER LE FICHIER DEPUIS VOS TÉLÉCHARGEMENTS ---")
# Ouvre la fenêtre pour sélectionner "Airplane Crashes and Fatalities upto 2023.zip"
uploaded = files.upload()

if uploaded:
    nom_fichier_zip = list(uploaded.keys())[0]
    print(f"\n✓ Fichier détecté en local : {nom_fichier_zip}")

    csv_file_inside = "Airplane_Crashes_and_Fatalities_Since_1908_t0_2023.csv"

    # Lecture directe de l'archive avec l'encodage correct pour les caractères spéciaux
    with zipfile.ZipFile(io.BytesIO(uploaded[nom_fichier_zip]), 'r') as z:
        with z.open(csv_file_inside) as f:
            df = pd.read_csv(io.BytesIO(f.read()), encoding='latin-1')

    print("\n--- DONNÉES RÉELLES CHARGÉES AVEC SUCCÈS ! ---")
    print(f"Dimensions du jeu de données : {df.shape[0]} lignes et {df.shape[1]} colonnes\n")

    # IDENTIFICATION PRÉALABLE DES VALEURS MANQUANTES (Exigence du Commentaire n°3)
    print("Nombre de valeurs manquantes réelles détectées par colonne :")
    print(df[['Date', 'Operator', 'Aboard', 'Fatalities', 'Ground']].isnull().sum())
    print("-" * 60)

    # =====================================================================
    # 2. NETTOYAGE ET IMPUTATION DES VALEURS MANQUANTES
    # =====================================================================
    # Traitement des chaînes vides cachées
    df.replace(['NULL', 'null', ' ', ''], np.nan, inplace=True)

    # Conversion de la colonne Date et nettoyage des lignes incorrectes
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df[df['Date'].notna()]
    df['Year'] = df['Date'].dt.year.astype(int)

    # Sécurisation des types numériques
    numeric_cols = ['Aboard', 'Fatalities', 'Ground']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Stratégie d'imputation par la médiane (Justifiée par la présence d'outliers)
    df['Fatalities'] = df['Fatalities'].fillna(df['Fatalities'].median())
    df['Aboard'] = df['Aboard'].fillna(df['Aboard'].median())
    df['Ground'] = df['Ground'].fillna(0)

    # Calcul du taux de survie par incident
    df['Survival_Rate'] = np.where(df['Aboard'] > 0, (df['Aboard'] - df['Fatalities']) / df['Aboard'], 0)
    df['Survival_Rate'] = df['Survival_Rate'].clip(0, 1)

    print("--- NETTOYAGE ET IMPUTATION TERMINÉS ---")
    print(f"Valeurs manquantes restantes dans 'Fatalities' : {df['Fatalities'].isnull().sum()}\n")

    # =====================================================================
    # 3. STATISTIQUES DESCRIPTIVES AVEC NUMPY & SCIPY (Jour 3)
    # =====================================================================
    print("--- ANALYSE STATISTIQUE DES DONNÉES ---")
    print(f"Nombre total de crashs analysés : {len(df)}")
    print(f"Total des décès cumulés : {int(df['Fatalities'].sum()):,}")

    mean_f = np.mean(df['Fatalities'])
    median_f = np.median(df['Fatalities'])
    skew_f = stats.skew(df['Fatalities'])

    print(f"Moyenne des décès par accident : {mean_f:.2f}")
    print(f"Médiane des décès par accident : {median_f:.2f}")
    print(f"Coefficient d'asymétrie (Skewness) : {skew_f:.2f}")

    # TEST D'HYPOTHÈSE : Comparaison de la mortalité inter-siècles
    siecle_20 = df[(df['Year'] >= 1908) & (df['Year'] < 2000)]['Fatalities']
    siecle_21 = df[(df['Year'] >= 2000) & (df['Year'] <= 2023)]['Fatalities']

    t_stat, p_value = stats.ttest_ind(siecle_20, siecle_21, equal_var=False)
    print(f"\nTest T de Student : t = {t_stat:.4f}, p-value = {p_value:.4e}")
    if p_value < 0.05:
        print("Conclusion : On rejette H0. La différence de mortalité moyenne entre le 20ème et au 21ème siècle est statistiquement significative.")
    else:
        print("Conclusion : On ne peut pas rejeter H0. Pas de différence statistique majeure.")

    # =====================================================================
    # 4. VISUALISATIONS GRAPHIQUES (Jour 4)
    # =====================================================================
    print("\n--- GÉNÉRATION DES GRAPHES D'ANALYSE ---")

    # Graphe 1 : Évolution du nombre annuel de crashs
    plt.figure(figsize=(12, 4))
    df_yearly = df.groupby('Year').size()
    plt.plot(df_yearly.index, df_yearly.values, color='darkblue', linewidth=2)
    plt.title("Évolution historique du nombre d'accidents d'avion par an (1908 - 2023)", fontsize=12)
    plt.xlabel("Années")
    plt.ylabel("Nombre de crashs")
    plt.show()

    # Graphe 2 : Distribution de la mortalité (Échelle Logarithmique)
    plt.figure(figsize=(12, 4))
    sns.histplot(df['Fatalities'], bins=50, kde=True, color='crimson')
    plt.yscale('log')
    plt.title("Distribution du nombre de décès par accident (Échelle Logarithmique)", fontsize=12)
    plt.xlabel("Nombre de victimes")
    plt.ylabel("Fréquence (Log)")
    plt.show()

else:
    print("Erreur : Aucun fichier sélectionné.")